# Terrain analysis: relative elevation model (REM)

Build a height above nearest drainage (HAND)-style relative elevation surface (DEM relative to a smoothed water surface) and summarize it by field (`rem_mean`) for subsequent stratification.

In [ ]:
# Shared color palette — used throughout all notebooks
STREAM_BLUE = "#2171b5"
WET_CMAP = "Blues"
TERRAIN_CMAP = "terrain"
REM_CMAP = "RdYlBu_r"
NDWI_CMAP = "RdYlGn"
STRATA_COLORS = {
    "perennial": "#2166ac",
    "intermittent": "#74add1",
    "managed": "#4dac26",
    "non_partitioned": "#d9d9d9",
}
PARTITION_COLORS = {"pe": "#a6cee3", "et_gwsm": "#1f78b4", "et_irr": "#e31a1c"}

## 1. Relative elevation model (REM)

For a normal REM one creates a continuous, sloping water surface profile that follows a specific river channel. It samples elevation points along a vector centerline and interpolates them across the valley bottom using Inverse Distance Weighting (IDW). Check out [RiverREM](https://github.com/OpenTopography/RiverREM) for a great example of this. This creates a single reference line against which the slope is detrended and the detail of the high resolution can be exploited.

With handily, we take this approach a little further to build evidence suggesting at what depth below the ground the water is. This involves a central assumption that the surface water is an expression of the elevation of the groundwater. This isn't a great assumption; in an area with groundwater wells pumping, the surface will be lower, and with groundwater recharge (from flood irrigation say), the water level might be higher. There are many hydrologic situations where there will be an elevational gradient to or away from the surface water naturally. In order to scale, we choose to live with this rather than go into the field and measure the depth to water using wells.

Compared to a normal REM, we're going to try and find more evidence of where the water is, taking into account potentially multiple surface water detections across the valley. We do this using both traditional hydrography GIS data (National Hydrographic Database) and remote sensing (aerial photography from the National Agricultural Imagery Program).

REM is computed as:

```
REM = DEM - Z_w
```

where `Z_w` is a locally smoothed estimate of water-surface elevation derived from a stream mask (NHD flowlines constrained by NDWI water pixels). Compared to routing-based HAND, this uses Gaussian smoothing and an observed water mask.

Low REM indicates near-channel settings with potential shallow groundwater/capillary support; high REM indicates terraces/uplands. The Beaverhead example uses `rem_mean < 2 m` as the default partitioning threshold; calibrate as needed.

## 2a. STAC/DEM acquisition

The DEM is fetched from a **local 3DEP STAC catalog** built with:

```bash
handily stac build --states MT --stac-dir ~/data/handily/stac/3dep_1m
```

`REMWorkflow` wraps three sequential steps. You can also call them individually for inspection:

```python
from handily.io import aoi_from_bounds
from handily.pipeline import REMWorkflow

aoi = aoi_from_bounds(bounds)
workflow = REMWorkflow(config, aoi)

workflow.fetch_vectors()   # clip NHD flowlines to AOI
workflow.fetch_dem()       # mosaic 3DEP tiles from local STAC
workflow.compute_rem()     # build stream mask + REM surface
```

The `fetch_dem()` call uses `stac_3dep.mosaic_from_stac()` to assemble overlapping LiDAR tiles into a single raster. All outputs are written to `config.out_dir`.

## 2. Setup and data loading

Load the project configuration and initialize the terrain workflow.

In [1]:
import os
import tomllib
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import rioxarray as rxr

from handily.config import HandilyConfig

# Load configuration
config_path = Path("beaverhead_config.toml")
with open(config_path, "rb") as f:
    cfg = tomllib.load(f)

config = HandilyConfig.from_dict(cfg)
bounds = tuple(cfg["bounds"])  # [W, S, E, N]

print(f"AOI bounds: {bounds}")
print(f"Output directory: {config.out_dir}")

AOI bounds: (-112.418, 45.445, -112.353, 45.49)
Output directory: /home/dgketchum/data/IrrigationGIS/handily/handily/beaverhead/outputs/


In [ ]:
# Hillshade + DEM overlay
dem_path = os.path.join(os.path.expanduser(config.out_dir), "dem_bounds_1m.tif")
if dem_path and os.path.exists(dem_path):
    import numpy as np
    dem_da = rxr.open_rasterio(dem_path).squeeze("band", drop=True)
    dem_arr = dem_da.values.astype(float)
    dzdx = np.gradient(dem_arr, axis=1)
    dzdy = np.gradient(dem_arr, axis=0)
    slope = np.sqrt(dzdx**2 + dzdy**2)
    aspect = np.arctan2(-dzdy, dzdx)
    az = np.radians(315)
    alt = np.radians(45)
    hillshade = np.sin(alt) * np.cos(slope) + np.cos(alt) * np.sin(slope) * np.cos(az - aspect)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(hillshade, cmap="gray")
    axes[0].set_title("Hillshade")
    dem_da.plot(ax=axes[1], cmap=TERRAIN_CMAP, robust=True)
    axes[1].set_title("DEM (m)")
    plt.tight_layout()
    plt.show()
else:
    print("[SKIP] DEM not yet computed — run REMWorkflow first")

## 3. Input data

### 3.1 Digital elevation model

USGS 3DEP 1 m LiDAR DEM accessed via a local STAC catalog. You can see in the range of 1430 - 1450 m elevation that the LiDAR data is capturing fantastic detail. The problem is that the entire valley is on a slope, descending from south to north.

In [2]:
# Load DEM if already computed, otherwise we'll compute it later
dem_path = os.path.join(os.path.expanduser(config.out_dir), "dem_bounds_1m.tif")

if os.path.exists(dem_path):
    dem = rxr.open_rasterio(dem_path).squeeze("band", drop=True)
    
    print(f"DEM shape: {dem.shape}")
    print(f"DEM CRS: {dem.rio.crs}")
    print(f"Resolution: {abs(dem.rio.resolution()[0]):.2f}m")
    print(f"Elevation range: {float(dem.min()):.1f} to {float(dem.max()):.1f}m")
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 8))
    dem.plot(ax=ax, cmap="terrain", robust=True)
    ax.set_title("Digital Elevation Model (1m LiDAR)")
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    plt.tight_layout()
else:
    print(f"DEM not found at {dem_path}")
    print("Run the REM workflow first to download the DEM.")

DEM not found at /home/dgketchum/data/IrrigationGIS/handily/handily/beaverhead/outputs/dem_bounds_1m.tif
Run the REM workflow first to download the DEM.


### 3.2 NHD flowlines

NHD flowlines provide channel geometry and stream type base on their FCODE attribute.

In [ ]:
# NDWI threshold sensitivity sweep
ndwi_path = os.path.join(os.path.expanduser(config.out_dir), "ndwi_bounds.tif")
if ndwi_path and os.path.exists(ndwi_path):
    ndwi_da = rxr.open_rasterio(ndwi_path).squeeze("band", drop=True)
    thresholds = [0.05, 0.10, 0.15, 0.20, 0.30]
    fig, axes = plt.subplots(1, len(thresholds), figsize=(18, 4))
    for ax, t in zip(axes, thresholds):
        (ndwi_da > t).plot(ax=ax, cmap=WET_CMAP, add_colorbar=False)
        ax.set_title(f"NDWI > {t}")
        ax.set_xlabel("")
        ax.set_ylabel("")
    plt.suptitle("NDWI Threshold Sensitivity")
    plt.tight_layout()
    plt.show()
else:
    print("[SKIP] NDWI not yet computed — run REMWorkflow first")

In [ ]:
# Flowlines overlaid on NDWI
ndwi_path = os.path.join(os.path.expanduser(config.out_dir), "ndwi_bounds.tif")
flowlines_path = os.path.join(os.path.expanduser(config.out_dir), "flowlines_bounds.fgb")
if os.path.exists(ndwi_path) and os.path.exists(flowlines_path):
    ndwi_da = rxr.open_rasterio(ndwi_path).squeeze("band", drop=True)
    flowlines_gdf = gpd.read_file(flowlines_path)
    fig, ax = plt.subplots(figsize=(10, 7))
    ndwi_da.plot(ax=ax, cmap=NDWI_CMAP, robust=True)
    flowlines_gdf.to_crs(ndwi_da.rio.crs).plot(ax=ax, color=STREAM_BLUE, linewidth=0.8, label="NHD flowlines")
    ax.set_title("NHD Flowlines over NDWI")
    ax.legend()
    plt.show()
else:
    print("[SKIP] NDWI or flowlines not yet computed — run REMWorkflow first")

In [3]:
# Load flowlines within the AOI
flowlines_path = os.path.join(os.path.expanduser(config.out_dir), "flowlines_bounds.fgb")

if os.path.exists(flowlines_path):
    flowlines = gpd.read_file(flowlines_path)
    
    print(f"Number of flowline segments: {len(flowlines)}")
    print(f"CRS: {flowlines.crs}")
    
    # Show FCODE distribution (stream types)
    if 'FCODE' in flowlines.columns:
        print("\nStream types (FCODE):")
        print(flowlines['FCODE'].value_counts().head(10))
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 8))
    flowlines.plot(ax=ax, linewidth=0.5, color='blue')
    ax.set_title("NHD Flowlines")
    ax.set_aspect('equal')
    plt.tight_layout()
else:
    print(f"Flowlines not found at {flowlines_path}")
    print("Run the REM workflow to extract flowlines.")

Flowlines not found at /home/dgketchum/data/IrrigationGIS/handily/handily/beaverhead/outputs/flowlines_bounds.fgb
Run the REM workflow to extract flowlines.


### 3.3 NDWI imagery

NDWI constrains the wetted-channel mask and reduces reliance on NHD-only rasterization.

In [4]:
# Load NDWI if available
ndwi_path = os.path.join(os.path.expanduser(config.out_dir), "ndwi_bounds.tif")

if os.path.exists(ndwi_path):
    ndwi = rxr.open_rasterio(ndwi_path).squeeze("band", drop=True)
    
    print(f"NDWI shape: {ndwi.shape}")
    print(f"NDWI range: {float(ndwi.min()):.2f} to {float(ndwi.max()):.2f}")
    
    # NDWI interpretation
    print("\nNDWI Interpretation:")
    print("  > 0.3: Water")
    print("  0.0 to 0.3: Wet vegetation or moist soil")
    print("  < 0.0: Dry land")
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Raw NDWI
    ndwi.plot(ax=axes[0], cmap="RdYlBu", vmin=-0.5, vmax=0.5)
    axes[0].set_title("NDWI Values")
    
    # Water mask at threshold
    threshold = cfg.get("ndwi_threshold", 0.15)
    water_mask = ndwi > threshold
    water_mask.plot(ax=axes[1], cmap="Blues")
    axes[1].set_title(f"Water Mask (NDWI > {threshold})")
    
    plt.tight_layout()
else:
    print(f"NDWI not found at {ndwi_path}")
    print("NDWI is loaded from ndwi_dir during the workflow.")

NDWI not found at /home/dgketchum/data/IrrigationGIS/handily/handily/beaverhead/outputs/ndwi_bounds.tif
NDWI is loaded from ndwi_dir during the workflow.


## 4. Stream mask

Combine rasterized NHD flowlines with an NDWI water mask to approximate wetted channels at DEM resolution:

```python
stream_mask = rasterized_flowlines & (ndwi > threshold)
```

This AND operation means a pixel is only marked as "stream" if:
1. It lies on an NHD flowline segment, **AND**
2. NDWI exceeds the threshold (actual water detected)

This filters out dry channel sections where NHD shows a flowline but no water is present.

In [5]:
# Visualize inputs to the AND operation: NDWI water mask (blue) + NHD flowlines (orange)
# Zoomed to a smaller AOI for detail
if os.path.exists(ndwi_path) and os.path.exists(flowlines_path):
    from rasterio import features
    from shapely.geometry import box
    import numpy as np
    
    # Zoom bounds (lat/lon)
    zoom_bounds = (-112.395, 45.465, -112.37, 45.480)  # W, S, E, N
    zoom_box = gpd.GeoDataFrame(geometry=[box(*zoom_bounds)], crs="EPSG:4326")
    zoom_box_proj = zoom_box.to_crs(ndwi.rio.crs)
    
    # Clip NDWI to zoom area
    ndwi_zoom = ndwi.rio.clip(zoom_box_proj.geometry, all_touched=True)
    
    # Clip flowlines to zoom area
    flowlines_proj = flowlines.to_crs(ndwi.rio.crs)
    flowlines_zoom = flowlines_proj.clip(zoom_box_proj)
    
    # Create NDWI water mask
    threshold = cfg.get("ndwi_threshold", 0.45)
    water_mask = (ndwi_zoom > threshold).astype("uint8")
    
    # Rasterize NHD flowlines to same grid
    transform = ndwi_zoom.rio.transform()
    shapes = [(geom, 1) for geom in flowlines_zoom.geometry if geom is not None]
    if shapes:
        nhd_raster = features.rasterize(
            shapes=shapes,
            out_shape=ndwi_zoom.shape,
            transform=transform,
            fill=0,
            all_touched=True,
            dtype="uint8",
        )
    else:
        nhd_raster = np.zeros(ndwi_zoom.shape, dtype="uint8")
    
    # Get extent for plotting
    extent = [
        float(ndwi_zoom.x.min()), float(ndwi_zoom.x.max()),
        float(ndwi_zoom.y.min()), float(ndwi_zoom.y.max())
    ]
    
    # Create figure
    fig, axes = plt.subplots(1, 3, figsize=(18, 8))
    
    # Left: NHD flowlines rasterized (orange)
    axes[0].imshow(nhd_raster, cmap="Oranges", extent=extent, origin="upper")
    axes[0].set_title("NHD Flowlines (rasterized)")
    axes[0].set_xlabel("Easting (m)")
    axes[0].set_aspect("equal")
    
    # Middle: NDWI water mask (blue)  
    axes[1].imshow(water_mask.values, cmap="Blues", extent=extent, origin="upper")
    axes[1].set_title(f"NDWI Water Mask (threshold={threshold})")
    axes[1].set_xlabel("Easting (m)")
    axes[1].set_aspect("equal")
    
    # Right: Overlay - Orange=NHD, Blue=NDWI, Green=Both (the AND result)
    rgb = np.zeros((*nhd_raster.shape, 3), dtype="uint8")
    rgb[..., 0] = nhd_raster * 255  # Red channel
    rgb[..., 1] = nhd_raster * 165  # Green channel (makes orange with red)
    rgb[..., 2] = water_mask.values * 255  # Blue channel = NDWI
    # Where both are true, we get white/cyan - override to green for clarity
    both_mask = (nhd_raster == 1) & (water_mask.values == 1)
    rgb[both_mask] = [0, 255, 0]  # Green where both
    
    axes[2].imshow(rgb, extent=extent, origin="upper")
    axes[2].set_title("Overlay: Orange=NHD only, Blue=NDWI only, Green=Both (AND)")
    axes[2].set_xlabel("Easting (m)")
    axes[2].set_aspect("equal")
    
    plt.tight_layout()
    
    # Stats
    nhd_pixels = nhd_raster.sum()
    ndwi_pixels = water_mask.values.sum()
    both_pixels = (nhd_raster & water_mask.values).sum()
    print(f"Zoom area: {zoom_bounds}")
    print(f"NHD flowline pixels: {nhd_pixels:,}")
    print(f"NDWI water pixels: {ndwi_pixels:,}")
    print(f"Both (AND result): {both_pixels:,}")
    if nhd_pixels > 0:
        print(f"Retention rate: {100 * both_pixels / nhd_pixels:.1f}% of NHD pixels have NDWI water")

### NHD buffering for misaligned flowlines

NHD flowlines are sometimes dated or misaligned with actual channel positions detected by NDWI. A buffer around the flowlines before rasterization captures nearby water pixels that would otherwise be lost to the AND operation.

In [6]:
# Compare unbuffered vs buffered NHD flowlines for the AND operation
if os.path.exists(ndwi_path) and os.path.exists(flowlines_path):
    from rasterio import features
    from shapely.geometry import box
    import numpy as np
    
    # Use same zoom bounds as previous cell
    zoom_bounds = (-112.395, 45.465, -112.37, 45.480)
    zoom_box = gpd.GeoDataFrame(geometry=[box(*zoom_bounds)], crs="EPSG:4326")
    zoom_box_proj = zoom_box.to_crs(ndwi.rio.crs)
    
    # Clip NDWI to zoom area
    ndwi_zoom = ndwi.rio.clip(zoom_box_proj.geometry, all_touched=True)
    
    # Clip flowlines to zoom area
    flowlines_proj = flowlines.to_crs(ndwi.rio.crs)
    flowlines_zoom = flowlines_proj.clip(zoom_box_proj)
    
    # NDWI water mask
    threshold = cfg.get("ndwi_threshold", 0.45)
    water_mask = (ndwi_zoom > threshold).astype("uint8")
    
    # Rasterize unbuffered flowlines
    transform = ndwi_zoom.rio.transform()
    shapes_unbuffered = [(geom, 1) for geom in flowlines_zoom.geometry if geom is not None]
    nhd_unbuffered = features.rasterize(
        shapes=shapes_unbuffered,
        out_shape=ndwi_zoom.shape,
        transform=transform,
        fill=0, all_touched=True, dtype="uint8",
    ) if shapes_unbuffered else np.zeros(ndwi_zoom.shape, dtype="uint8")
    
    # Rasterize buffered flowlines (10m buffer as in config)
    buffer_m = cfg.get("flowlines_buffer_m", 10.0)
    flowlines_buffered = flowlines_zoom.copy()
    flowlines_buffered["geometry"] = flowlines_buffered.geometry.buffer(buffer_m)
    shapes_buffered = [(geom, 1) for geom in flowlines_buffered.geometry if geom is not None]
    nhd_buffered = features.rasterize(
        shapes=shapes_buffered,
        out_shape=ndwi_zoom.shape,
        transform=transform,
        fill=0, all_touched=True, dtype="uint8",
    ) if shapes_buffered else np.zeros(ndwi_zoom.shape, dtype="uint8")
    
    # Compute AND results
    and_unbuffered = nhd_unbuffered & water_mask.values
    and_buffered = nhd_buffered & water_mask.values
    
    # Get extent for plotting
    extent = [
        float(ndwi_zoom.x.min()), float(ndwi_zoom.x.max()),
        float(ndwi_zoom.y.min()), float(ndwi_zoom.y.max())
    ]
    
    # Create comparison figure
    fig, axes = plt.subplots(1, 3, figsize=(18, 8))
    
    # Left: Unbuffered AND result
    axes[0].imshow(and_unbuffered, cmap="Greens", extent=extent, origin="upper")
    axes[0].set_title(f"AND Result: Unbuffered NHD\n({and_unbuffered.sum():,} pixels)")
    axes[0].set_xlabel("Easting (m)")
    axes[0].set_aspect("equal")
    
    # Middle: Buffered AND result
    axes[1].imshow(and_buffered, cmap="Greens", extent=extent, origin="upper")
    axes[1].set_title(f"AND Result: {buffer_m}m Buffer\n({and_buffered.sum():,} pixels)")
    axes[1].set_xlabel("Easting (m)")
    axes[1].set_aspect("equal")
    
    # Right: Comparison overlay - show pixels gained by buffering
    rgb = np.zeros((*nhd_unbuffered.shape, 3), dtype="uint8")
    # Green = both methods capture this pixel
    rgb[and_unbuffered == 1] = [0, 200, 0]
    # Cyan = only buffered method captures this (gained pixels)
    gained = (and_buffered == 1) & (and_unbuffered == 0)
    rgb[gained] = [0, 255, 255]
    
    axes[2].imshow(rgb, extent=extent, origin="upper")
    axes[2].set_title(f"Comparison: Green=Both, Cyan=Gained with buffer\n({gained.sum():,} additional pixels)")
    axes[2].set_xlabel("Easting (m)")
    axes[2].set_aspect("equal")
    
    plt.tight_layout()
    
    # Summary stats
    print(f"Buffer distance: {buffer_m}m")
    print(f"NDWI water pixels in zoom: {water_mask.values.sum():,}")
    print(f"Unbuffered AND: {and_unbuffered.sum():,} pixels")
    print(f"Buffered AND: {and_buffered.sum():,} pixels")
    print(f"Improvement: +{and_buffered.sum() - and_unbuffered.sum():,} pixels ({100*(and_buffered.sum() - and_unbuffered.sum())/max(1, and_unbuffered.sum()):.1f}% increase)")

In [7]:
# Load streams mask if available
streams_path = os.path.join(os.path.expanduser(config.out_dir), "streams_bounds.tif")

if os.path.exists(streams_path):
    streams = rxr.open_rasterio(streams_path).squeeze("band", drop=True)
    
    n_stream_pixels = int((streams == 1).sum())
    total_pixels = streams.size
    pct_streams = 100 * n_stream_pixels / total_pixels
    
    print(f"Stream pixels: {n_stream_pixels:,} ({pct_streams:.2f}%)")
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 8))
    streams.plot(ax=ax, cmap="Blues")
    ax.set_title("Stream Mask (NHD + NDWI)")
    plt.tight_layout()
else:
    print(f"Streams mask not found at {streams_path}")

Streams mask not found at /home/dgketchum/data/IrrigationGIS/handily/handily/beaverhead/outputs/streams_bounds.tif


In [ ]:
# REM distribution histogram with quantiles
import numpy as np
rem_path = os.path.join(os.path.expanduser(config.out_dir), "rem_bounds.tif")
if os.path.exists(rem_path):
    rem_da = rxr.open_rasterio(rem_path).squeeze("band", drop=True)
    vals = rem_da.values.flatten()
    vals = vals[~np.isnan(vals) & (vals > 0) & (vals < 20)]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(vals, bins=80, color=STREAM_BLUE, edgecolor="none", alpha=0.7)
    for q, label in [(0.25, "Q1"), (0.5, "Median"), (0.75, "Q3")]:
        v = float(np.quantile(vals, q))
        ax.axvline(v, color="red", linestyle="--", label=f"{label}: {v:.2f} m")
    ax.axvline(config.rem_threshold, color="orange", linewidth=2,
               label=f"Partition threshold: {config.rem_threshold} m")
    ax.set_xlabel("REM (m)")
    ax.set_ylabel("Pixel count")
    ax.set_title("REM Distribution")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("[SKIP] REM not yet computed — run REMWorkflow first")

### NDWI threshold

Tune to trade omission vs commission (wet vegetation, shadow, and turbid water). The Beaverhead example uses `0.45`.

## 5. Computing the REM

1. Sample DEM elevations along the stream mask.
2. Gaussian smooth stream elevations to a continuous water-surface proxy (`Z_w`).
3. Compute `REM = DEM - Z_w`.

The smoothing radius controls how far the influence of streams is extended across the floodplain.

In [8]:
# Load REM if available
rem_path = os.path.join(os.path.expanduser(config.out_dir), "rem_bounds.tif")

if os.path.exists(rem_path):
    rem = rxr.open_rasterio(rem_path).squeeze("band", drop=True)
    
    print(f"REM shape: {rem.shape}")
    print(f"REM range: {float(rem.min()):.1f} to {float(rem.max()):.1f}m")
    print(f"REM mean: {float(rem.mean()):.1f}m")
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Full range
    rem.plot(ax=axes[0], cmap="RdYlBu_r", robust=True)
    axes[0].set_title("Relative Elevation Model (REM)")
    
    # Focus on low elevations (0-10m)
    rem.plot(ax=axes[1], cmap="RdYlBu_r", vmin=0, vmax=10)
    axes[1].set_title("REM (0-10m range)")
    
    plt.tight_layout()
else:
    print(f"REM not found at {rem_path}")

REM not found at /home/dgketchum/data/IrrigationGIS/handily/handily/beaverhead/outputs/rem_bounds.tif


### Interpreting the REM

| REM Value | Landscape position | Shallow-GW influence |
|-----------|--------------------|----------------------|
| 0﷿﷿﷿1 m | Near water surface | High |
| 1﷿﷿﷿2 m | Low terrace/floodplain | Moderate﷿﷿﷿high |
| 2﷿﷿﷿5 m | Intermediate terrace | Moderate |
| 5﷿﷿﷿10 m | High terrace | Low |
| >10 m | Upland | Very low |

### Operational Callout — `handily bounds`

The CLI `handily bounds` runs the full REM pipeline within a bounding box without writing Python:

```bash
handily bounds --bounds -112.42 45.44 -112.35 45.50 \
  --fields fields.gpkg --ndwi-dir /data/ndwi \
  --flowlines-local-dir /data/nhd --stac-dir /data/stac \
  --out-dir /tmp/bvr
```

### Operational Callout — `handily aoi`

For statewide or multi-watershed runs, `handily aoi` tiles a field layer into spatially-manageable AOI polygons before running `bounds` per tile:

```bash
handily aoi --fields fields.gpkg --out aoi_tiles.fgb
```

> **Repo Capability — Notebook 2 (Terrain Analysis)**
> Modules exercised: `handily.pipeline`, `handily.compute`, `handily.stac_3dep`, `handily.dem`, `handily.nhd`
> Key functions: `REMWorkflow.run()`, `REMWorkflow.fetch_dem()`, `REMWorkflow.compute_rem()`, `build_streams_mask_from_nhd_ndwi()`, `compute_rem_quick()`
> CLI equivalents: `handily bounds`, `handily stac build/extend`, `handily aoi`

> **Artifacts Produced**
> - `{out_dir}/dem_bounds_1m.tif` — 1 m LiDAR DEM clipped to AOI bounds
> - `{out_dir}/flowlines_bounds.fgb` — NHD flowlines clipped to AOI
> - `{out_dir}/streams_bounds.tif` — Combined NHD + NDWI stream mask
> - `{out_dir}/ndwi_bounds.tif` — NDWI mosaic clipped to AOI
> - `{out_dir}/rem_bounds.tif` — Relative Elevation Model
> - `{out_dir}/fields_bounds.fgb` — Field polygons with `rem_mean` column

## 6. Field statistics

Summarize REM by field polygon (mean and distributional stats). `rem_mean` is used for partitioned/non-partitioned classification in the example workflow.

In [9]:
# Load fields with REM statistics
fields_path = os.path.join(os.path.expanduser(config.out_dir), "fields_bounds.fgb")

if os.path.exists(fields_path):
    fields = gpd.read_file(fields_path)
    
    print(f"Number of fields: {len(fields)}")
    print(f"Columns: {list(fields.columns)}")
    
    if 'rem_mean' in fields.columns:
        print("\nREM Statistics:")
        print(f"  Min mean REM: {fields['rem_mean'].min():.2f}m")
        print(f"  Max mean REM: {fields['rem_mean'].max():.2f}m")
        print(f"  Median mean REM: {fields['rem_mean'].median():.2f}m")
        
        # Histogram
        fig, ax = plt.subplots(figsize=(10, 5))
        fields['rem_mean'].hist(ax=ax, bins=30, edgecolor='black')
        ax.axvline(2.0, color='red', linestyle='--', label='Threshold (2m)')
        ax.set_xlabel("Mean REM (m)")
        ax.set_ylabel("Number of Fields")
        ax.set_title("Distribution of Field Mean REM")
        ax.legend()
        plt.tight_layout()
    else:
        print("rem_mean column not found. Run the REM workflow.")
else:
    print(f"Fields not found at {fields_path}")

Fields not found at /home/dgketchum/data/IrrigationGIS/handily/handily/beaverhead/outputs/fields_bounds.fgb


In [ ]:
from handily.io import aoi_from_bounds
from handily.pipeline import REMWorkflow

aoi = aoi_from_bounds(bounds)

if config.run_rem:
    workflow = REMWorkflow(config, aoi)
    results = workflow.run(
        ndwi_threshold=config.ndwi_threshold,
        flowlines_buffer_m=config.flowlines_buffer_m,
    )
    print("REM workflow complete:", list(results.keys()))
else:
    print("[SKIP] run_rem=False in config — load cached outputs below")

In [12]:
# Example: Running the REM workflow
# (Commented out to avoid re-running if already complete)

# from handily.io import aoi_from_bounds, ensure_dir
# from handily.pipeline import REMWorkflow
#
# # Create AOI from bounds
# aoi = aoi_from_bounds(bounds)
#
# # Initialize workflow
# workflow = REMWorkflow(config=config, aoi=aoi)
#
# # Run the workflow
# # This fetches vectors, downloads DEM, and computes REM
# result = workflow.run(
#     ndwi_threshold=cfg.get('ndwi_threshold', 0.15),
#     stats=('mean',),
#     cache_flowlines=True
# )
#
# # Access results
# print(f"Fields processed: {result['summary']['total_fields']}")
# print(f"DEM shape: {result['dem'].shape}")
# print(f"REM range: {float(result['rem'].min()):.1f} to {float(result['rem'].max()):.1f}m")

print("See beaverhead.py for the full workflow script.")

See beaverhead.py for the full workflow script.


### Workflow output files

| File | Description |
|------|-------------|
| `dem_bounds_1m.tif` | 1 m DEM clipped to AOI |
| `flowlines_bounds.fgb` | NHD flowlines within AOI |
| `streams_bounds.tif` | Combined stream mask |
| `ndwi_bounds.tif` | NDWI mosaic clipped to AOI |
| `rem_bounds.tif` | Relative elevation model |
| `fields_bounds.fgb` | Fields with `rem_mean` column |

## Key takeaways

- REM is a practical proxy for shallow-GW access at field scale.
- Constraining NHD flowlines with NDWI improves the wetted-channel mask.
- `rem_mean` provides a simple stratification metric; thresholds are study-specific.

**Next**: [Notebook 03 - Field Classification](03_field_classification.ipynb)